In [9]:
# Cài đặt thư viện
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier

In [ ]:
# Đọc file
ht = pd.read_csv(r"c:\Users\USER\Downloads\ClassifyRisk.csv")
ht

,mortgage,loans,age,marital_status,income,risk
0,y,3,34,other,28060.70,bad loss
1,n,2,37,other,28009.34,bad loss
2,n,2,29,other,27614.60,bad loss
3,y,2,33,other,27287.18,bad loss
4,y,2,39,other,26954.06,bad loss
...,...,...,...,...,...,...
241,y,0,51,married,46810.12,good risk
242,y,0,55,married,45709.78,good risk
243,y,0,51,married,44896.42,good risk
244,y,0,54,married,44301.52,good risk


In [16]:
# Kiểm tra định dạng dữ liệu
ht.info()
ht.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   mortgage        246 non-null    object 
 1   loans           246 non-null    int64  
 2   age             246 non-null    int64  
 3   marital_status  246 non-null    object 
 4   income          246 non-null    float64
 5   risk            246 non-null    object 
dtypes: float64(1), int64(2), object(3)
memory usage: 11.7+ KB


mortgage          0
loans             0
age               0
marital_status    0
income            0
risk              0
dtype: int64

In [20]:
X = ht.drop(columns=["risk"])
y = ht["risk"]


In [24]:
# Encode biến mục tiêu risk: bad loss / good risk
le = LabelEncoder()
y = le.fit_transform(y)

# Xử lý các cột chữ trong X bằng one-hot encoding
X = pd.get_dummies(X, drop_first=True)

print(X.head())
print(le.classes_)

   loans  age    income  mortgage_y  marital_status_other  \
0      3   34  28060.70        True                  True   
1      2   37  28009.34       False                  True   
2      2   29  27614.60       False                  True   
3      2   33  27287.18        True                  True   
4      2   39  26954.06        True                  True   

   marital_status_single  
0                  False  
1                  False  
2                  False  
3                  False  
4                  False  
[0 1]


In [ ]:
# =========================
# 6. CHIA TRAIN / TEST 70 / 30
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (172, 6)
Test size: (74, 6)


In [23]:
# =========================
# 7. DECISION TREE
# =========================

dt_params = {
    "criterion": ["gini", "entropy"],
    "ccp_alpha": [0, 0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001]
}

dt_model = DecisionTreeClassifier(random_state=42)

dt_grid = GridSearchCV(
    estimator=dt_model,
    param_grid=dt_params,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

best_dt = dt_grid.best_estimator_
dt_pred = best_dt.predict(X_test)

dt_acc = accuracy_score(y_test, dt_pred)

print("===== DECISION TREE =====")
print("Best params:", dt_grid.best_params_)
print("Accuracy:", dt_acc)
print("Confusion matrix:")
print(confusion_matrix(y_test, dt_pred))
print("Classification report:")
print(classification_report(y_test, dt_pred, target_names=le.classes_))

===== DECISION TREE =====
Best params: {'ccp_alpha': 0.01, 'criterion': 'gini'}
Accuracy: 0.918918918918919
Confusion matrix:
[[35  2]
 [ 4 33]]
Classification report:
              precision    recall  f1-score   support

    bad loss       0.90      0.95      0.92        37
   good risk       0.94      0.89      0.92        37

    accuracy                           0.92        74
   macro avg       0.92      0.92      0.92        74
weighted avg       0.92      0.92      0.92        74



In [26]:
# =========================
# 8. BAGGING CLASSIFIER
# =========================

bagging_params = {
    "n_estimators": [10, 50, 100],
    "max_samples": [0.5, 0.7, 1.0],
    "max_features": [0.5, 0.7, 1.0]
}

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    random_state=42
)

bagging_grid = GridSearchCV(
    estimator=bagging_model,
    param_grid=bagging_params,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

bagging_grid.fit(X_train, y_train)

best_bagging = bagging_grid.best_estimator_
bagging_pred = best_bagging.predict(X_test)

bagging_acc = accuracy_score(y_test, bagging_pred)

print("===== BAGGING CLASSIFIER =====")
print("Best params:", bagging_grid.best_params_)
print("Accuracy:", bagging_acc)
print("Confusion matrix:")
print(confusion_matrix(y_test, bagging_pred))
print("Classification report:")
print(classification_report(
    y_test,
    bagging_pred,
    target_names=["bad loss", "good risk"]
))

===== BAGGING CLASSIFIER =====
Best params: {'max_features': 0.5, 'max_samples': 0.7, 'n_estimators': 50}
Accuracy: 0.918918918918919
Confusion matrix:
[[34  3]
 [ 3 34]]
Classification report:
              precision    recall  f1-score   support

    bad loss       0.92      0.92      0.92        37
   good risk       0.92      0.92      0.92        37

    accuracy                           0.92        74
   macro avg       0.92      0.92      0.92        74
weighted avg       0.92      0.92      0.92        74



In [28]:
# =========================
# 9. RANDOM FOREST
# =========================

rf_params = {
    "n_estimators": [50, 100, 200],
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 3, 5, 10],
    "ccp_alpha": [0, 0.001, 0.005, 0.01]
}

rf_model = RandomForestClassifier(random_state=42)

rf_grid = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_params,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
rf_pred = best_rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)

print("===== RANDOM FOREST =====")
print("Best params:", rf_grid.best_params_)
print("Accuracy:", rf_acc)
print("Confusion matrix:")
print(confusion_matrix(y_test, rf_pred))
print("Classification report:")
print(classification_report(
    y_test,
    bagging_pred,
    target_names=["bad loss", "good risk"]
))

===== RANDOM FOREST =====
Best params: {'ccp_alpha': 0, 'criterion': 'entropy', 'max_depth': None, 'n_estimators': 50}
Accuracy: 0.9324324324324325
Confusion matrix:
[[35  2]
 [ 3 34]]
Classification report:
              precision    recall  f1-score   support

    bad loss       0.92      0.92      0.92        37
   good risk       0.92      0.92      0.92        37

    accuracy                           0.92        74
   macro avg       0.92      0.92      0.92        74
weighted avg       0.92      0.92      0.92        74



In [29]:
# =========================
# 10. GRADIENT BOOSTING
# =========================

gb_params = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 5]
}

gb_model = GradientBoostingClassifier(random_state=42)

gb_grid = GridSearchCV(
    estimator=gb_model,
    param_grid=gb_params,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

gb_grid.fit(X_train, y_train)

best_gb = gb_grid.best_estimator_
gb_pred = best_gb.predict(X_test)

gb_acc = accuracy_score(y_test, gb_pred)

print("===== GRADIENT BOOSTING =====")
print("Best params:", gb_grid.best_params_)
print("Accuracy:", gb_acc)
print("Confusion matrix:")
print(confusion_matrix(y_test, gb_pred))
print("Classification report:")
print(classification_report(
    y_test,
    bagging_pred,
    target_names=["bad loss", "good risk"]
))

===== GRADIENT BOOSTING =====
Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
Accuracy: 0.9324324324324325
Confusion matrix:
[[34  3]
 [ 2 35]]
Classification report:
              precision    recall  f1-score   support

    bad loss       0.92      0.92      0.92        37
   good risk       0.92      0.92      0.92        37

    accuracy                           0.92        74
   macro avg       0.92      0.92      0.92        74
weighted avg       0.92      0.92      0.92        74



In [30]:
# =========================
# 11. TỔNG HỢP KẾT QUẢ
# =========================

results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Bagging Classifier",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Accuracy": [
        dt_acc,
        bagging_acc,
        rf_acc,
        gb_acc
    ],
    "Best Params": [
        dt_grid.best_params_,
        bagging_grid.best_params_,
        rf_grid.best_params_,
        gb_grid.best_params_
    ]
})

results = results.sort_values(by="Accuracy", ascending=False)

results

,Model,Accuracy,Best Params
3,Gradient Boosting,0.932432,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est..."
2,Random Forest,0.932432,"{'ccp_alpha': 0, 'criterion': 'entropy', 'max_..."
1,Bagging Classifier,0.918919,"{'max_features': 0.5, 'max_samples': 0.7, 'n_e..."
0,Decision Tree,0.918919,"{'ccp_alpha': 0.01, 'criterion': 'gini'}"


In [31]:
# =========================
# 12. MÔ HÌNH TỐT NHẤT
# =========================

best_model_name = results.iloc[0]["Model"]
best_accuracy = results.iloc[0]["Accuracy"]
best_params = results.iloc[0]["Best Params"]

print("Mô hình tốt nhất là:", best_model_name)
print("Accuracy:", best_accuracy)
print("Best params:", best_params)

Mô hình tốt nhất là: Gradient Boosting
Accuracy: 0.9324324324324325
Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
